In [1]:
print("Jupyter is working")

Jupyter is working


In [2]:
%pip install pandas numpy scikit-learn matplotlib seaborn xgboost networkx shap river streamlit torch torch-geometric

Note: you may need to restart the kernel to use updated packages.


In [3]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import pandas
import numpy
import sklearn
import matplotlib
import seaborn
import xgboost
import networkx
import shap
import river
import streamlit
import torch
import torch_geometric

print("All required libraries installed successfully!")

All required libraries installed successfully!


In [7]:
import pandas as pd

df = pd.read_csv("paysim.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (6362620, 11)


In [5]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nFraud distribution:")
print(df["isFraud"].value_counts())

print("\nTransaction types:")
print(df["type"].value_counts())

Missing values:
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

Duplicate rows: 0

Fraud distribution:
isFraud
0    6354407
1       8213
Name: count, dtype: int64

Transaction types:
type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64


In [6]:
print("Duplicate rows:", df.duplicated().sum())

print("\nFraud distribution:")
print(df["isFraud"].value_counts())

print("\nTransaction types:")
print(df["type"].value_counts())

Duplicate rows: 0

Fraud distribution:
isFraud
0    6354407
1       8213
Name: count, dtype: int64

Transaction types:
type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64


In [7]:
fraud_by_type = pd.crosstab(df["type"], df["isFraud"])

print(fraud_by_type)

isFraud         0     1
type                   
CASH_IN   1399284     0
CASH_OUT  2233384  4116
DEBIT       41432     0
PAYMENT   2151495     0
TRANSFER   528812  4097


In [8]:
print("Numerical feature summary:")
print(df[[
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]].describe())

Numerical feature summary:
             amount  oldbalanceOrg  newbalanceOrig  oldbalanceDest  \
count  6.362620e+06   6.362620e+06    6.362620e+06    6.362620e+06   
mean   1.798619e+05   8.338831e+05    8.551137e+05    1.100702e+06   
std    6.038582e+05   2.888243e+06    2.924049e+06    3.399180e+06   
min    0.000000e+00   0.000000e+00    0.000000e+00    0.000000e+00   
25%    1.338957e+04   0.000000e+00    0.000000e+00    0.000000e+00   
50%    7.487194e+04   1.420800e+04    0.000000e+00    1.327057e+05   
75%    2.087215e+05   1.073152e+05    1.442584e+05    9.430367e+05   
max    9.244552e+07   5.958504e+07    4.958504e+07    3.560159e+08   

       newbalanceDest  
count    6.362620e+06  
mean     1.224996e+06  
std      3.674129e+06  
min      0.000000e+00  
25%      0.000000e+00  
50%      2.146614e+05  
75%      1.111909e+06  
max      3.561793e+08  


In [9]:
fraud_count = df["isFraud"].sum()
total_count = len(df)

fraud_rate = (fraud_count / total_count) * 100

print("Total transactions:", total_count)
print("Fraud transactions:", fraud_count)
print("Normal transactions:", total_count - fraud_count)
print("Fraud rate:", round(fraud_rate, 4), "%")

Total transactions: 6362620
Fraud transactions: 8213
Normal transactions: 6354407
Fraud rate: 0.1291 %


In [10]:
fraud_amount = df[df["isFraud"] == 1]["amount"].mean()
normal_amount = df[df["isFraud"] == 0]["amount"].mean()

print("Average fraud transaction amount:", round(fraud_amount, 2))
print("Average normal transaction amount:", round(normal_amount, 2))

Average fraud transaction amount: 1467967.3
Average normal transaction amount: 178197.04


In [11]:
print("Minimum step:", df["step"].min())
print("Maximum step:", df["step"].max())
print("Unique time steps:", df["step"].nunique())

Minimum step: 1
Maximum step: 743
Unique time steps: 743


In [12]:
print("Step is chronological:", df["step"].is_monotonic_increasing)

Step is chronological: True


In [13]:
features = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

X = df[features].copy()
y = df["isFraud"].copy()

X = pd.get_dummies(X, columns=["type"], dtype=int)

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeatures:")
print(X.columns.tolist())

Feature shape: (6362620, 11)
Target shape: (6362620,)

Features:
['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']


In [14]:
split_index = int(len(X) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("Training frauds:", y_train.sum())
print("Testing frauds:", y_test.sum())

Training data: (5090096, 11)
Testing data: (1272524, 11)
Training frauds: 3959
Testing frauds: 4254


In [15]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    eval_metric="logloss",
    random_state=42
)

xgb_model.fit(X_train, y_train)

print("XGBoost model trained successfully!")

XGBoost model trained successfully!


In [16]:
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:, 1]

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
pr_auc = average_precision_score(y_test, y_prob)

print("XGBoost Performance")
print("--------------------")
print("Precision:", precision)
print("Recall:", recall)
print("F1-Score:", f1)
print("PR-AUC:", pr_auc)

XGBoost Performance
--------------------
Precision: 0.9987046632124352
Recall: 0.7249647390691114
F1-Score: 0.840098065922092
PR-AUC: 0.9623661187973616


In [17]:
import joblib

joblib.dump(xgb_model, "xgb_fraud_model.pkl")

print("XGBoost model saved successfully!")

XGBoost model saved successfully!


In [8]:
tgn_df = df.iloc[:100000].copy()

tgn_df["origin_tx_count"] = tgn_df.groupby("nameOrig").cumcount()
tgn_df["dest_tx_count"] = tgn_df.groupby("nameDest").cumcount()

tgn_df["origin_prev_amount"] = (
    tgn_df.groupby("nameOrig")["amount"]
    .shift(1)
    .fillna(0)
)

tgn_df["dest_prev_amount"] = (
    tgn_df.groupby("nameDest")["amount"]
    .shift(1)
    .fillna(0)
)

tgn_df["origin_unique_dest"] = (
    tgn_df.groupby("nameOrig")["nameDest"]
    .transform("nunique")
)

tgn_df["dest_unique_orig"] = (
    tgn_df.groupby("nameDest")["nameOrig"]
    .transform("nunique")
)

print("tgn_df created:", tgn_df.shape)

tgn_df created: (100000, 17)


In [9]:
import networkx as nx

G = nx.DiGraph()

for _, row in tgn_df.iterrows():
    G.add_edge(
        row["nameOrig"],
        row["nameDest"],
        amount=row["amount"],
        step=row["step"],
        isFraud=row["isFraud"]
    )

print("Graph created!")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Graph created!
Nodes: 151551
Edges: 100000


In [10]:
components = list(nx.weakly_connected_components(G))

fraud_nodes = set(
    tgn_df.loc[tgn_df["isFraud"] == 1, "nameOrig"]
).union(
    set(tgn_df.loc[tgn_df["isFraud"] == 1, "nameDest"])
)

fraud_communities = [
    community for community in components
    if community.intersection(fraud_nodes)
]

print("Fraud-containing communities:", len(fraud_communities))

Fraud-containing communities: 115


In [11]:
fraud_nodes = set(
    tgn_df.loc[tgn_df["isFraud"] == 1, "nameOrig"]
).union(
    set(tgn_df.loc[tgn_df["isFraud"] == 1, "nameDest"])
)

components = list(nx.weakly_connected_components(G))

fraud_communities = [
    community for community in components
    if community.intersection(fraud_nodes)
]

print("Fraud-containing communities:", len(fraud_communities))

Fraud-containing communities: 115


In [12]:
fraud_ring_summary = []

for ring_id, community in enumerate(fraud_communities, start=1):

    ring_transactions = tgn_df[
        tgn_df["nameOrig"].isin(community) |
        tgn_df["nameDest"].isin(community)
    ]

    transactions = len(ring_transactions)
    total_amount = ring_transactions["amount"].sum()
    fraud_transactions = ring_transactions["isFraud"].sum()

    fraud_ratio = (
        fraud_transactions / transactions
        if transactions > 0
        else 0
    )

    fraud_ring_summary.append({
        "Ring ID": ring_id,
        "Accounts in ring": len(community),
        "Transactions": transactions,
        "Total amount": total_amount,
        "Fraud ratio": fraud_ratio
    })

fraud_ring_summary = pd.DataFrame(fraud_ring_summary)

fraud_ring_summary = fraud_ring_summary.sort_values(
    by=["Fraud ratio", "Transactions"],
    ascending=False
).reset_index(drop=True)

print("Fraud-ring summary created successfully!")
print(fraud_ring_summary.head(15))

fraud_ring_summary.to_csv(
    "fraud_ring_summary.csv",
    index=False
)

print("\nSaved as fraud_ring_summary.csv")

Fraud-ring summary created successfully!
    Ring ID  Accounts in ring  Transactions  Total amount  Fraud ratio
0        16                 2             1      35063.63          1.0
1        28                 2             1    1096187.24          1.0
2        29                 2             1     963532.14          1.0
3        44                 2             1     120074.73          1.0
4        78                 2             1        164.00          1.0
5        79                 2             1      21571.00          1.0
6        92                 2             1      89631.24          1.0
7        94                 2             1      29707.86          1.0
8       108                 2             1      21922.00          1.0
9       110                 2             1       1055.00          1.0
10      111                 2             1     244068.01          1.0
11      112                 2             1      59835.00          1.0
12      113                 2       

In [23]:
fraud_ring_summary.to_csv("fraud_ring_summary.csv", index=False)

print("Fraud-ring summary saved successfully!")

Fraud-ring summary saved successfully!


In [24]:
fraud_edge_count = tgn_df["isFraud"].sum()

fraud_related_nodes = set(
    tgn_df.loc[tgn_df["isFraud"] == 1, "nameOrig"]
).union(
    set(tgn_df.loc[tgn_df["isFraud"] == 1, "nameDest"])
)

print("Fraud transactions in graph:", fraud_edge_count)
print("Fraud-related nodes:", len(fraud_related_nodes))

Fraud transactions in graph: 116
Fraud-related nodes: 231


In [25]:
degrees = dict(G.degree())

top_nodes = sorted(
    degrees.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]

print("Top 10 highly connected nodes:")
for node, degree in top_nodes:
    print(node, "->", degree)

Top 10 highly connected nodes:
C985934102 -> 78
C1286084959 -> 72
C248609774 -> 71
C1590550415 -> 69
C2083562754 -> 66
C977993101 -> 65
C665576141 -> 65
C1360767589 -> 62
C451111351 -> 60
C97730845 -> 57


In [26]:
avg_degree = sum(degrees.values()) / len(degrees)
max_degree = max(degrees.values())

print("Average node degree:", round(avg_degree, 4))
print("Maximum node degree:", max_degree)

Average node degree: 1.3197
Maximum node degree: 78


In [27]:
graph_summary = {
    "total_nodes": G.number_of_nodes(),
    "total_edges": G.number_of_edges(),
    "total_communities": len(components),
    "largest_community_size": largest_community,
    "fraud_containing_communities": len(fraud_communities),
    "fraud_transactions": int(fraud_edge_count),
    "fraud_related_nodes": len(fraud_related_nodes),
    "average_node_degree": round(avg_degree, 4),
    "maximum_node_degree": max_degree
}

graph_summary_df = pd.DataFrame([graph_summary])
graph_summary_df.to_csv("graph_summary.csv", index=False)

print("Graph analysis results saved successfully!")
print(graph_summary_df)

Graph analysis results saved successfully!
   total_nodes  total_edges  total_communities  largest_community_size  \
0       151551       100000              51551                      79   

   fraud_containing_communities  fraud_transactions  fraud_related_nodes  \
0                           115                 116                  231   

   average_node_degree  maximum_node_degree  
0               1.3197                   78  


In [28]:
top_nodes_df = pd.DataFrame(
    top_nodes,
    columns=["node", "degree"]
)

top_nodes_df.to_csv("top_connected_nodes.csv", index=False)

print("Top connected nodes saved successfully!")

Top connected nodes saved successfully!


In [29]:
top_node_fraud = []

for node, degree in top_nodes:
    fraud_outgoing = tgn_df[
        (tgn_df["nameOrig"] == node) &
        (tgn_df["isFraud"] == 1)
    ].shape[0]

    top_node_fraud.append({
        "node": node,
        "degree": degree,
        "fraud_outgoing_transactions": fraud_outgoing
    })

top_node_fraud_df = pd.DataFrame(top_node_fraud)

print(top_node_fraud_df)

          node  degree  fraud_outgoing_transactions
0   C985934102      78                            0
1  C1286084959      72                            0
2   C248609774      71                            0
3  C1590550415      69                            0
4  C2083562754      66                            0
5   C977993101      65                            0
6   C665576141      65                            0
7  C1360767589      62                            0
8   C451111351      60                            0
9    C97730845      57                            0


In [30]:
top_node_fraud_df.to_csv("top_connected_node_fraud_analysis.csv", index=False)

print("Top-node fraud analysis saved successfully!")

Top-node fraud analysis saved successfully!


In [31]:
graph_dashboard_data = {
    "Total Nodes": G.number_of_nodes(),
    "Total Edges": G.number_of_edges(),
    "Total Communities": len(components),
    "Largest Community": largest_community,
    "Fraud Communities": len(fraud_communities),
    "Fraud Transactions": int(fraud_edge_count),
    "Fraud-Related Nodes": len(fraud_related_nodes),
    "Average Degree": round(avg_degree, 4),
    "Maximum Degree": max_degree
}

graph_dashboard_df = pd.DataFrame(
    list(graph_dashboard_data.items()),
    columns=["Metric", "Value"]
)

graph_dashboard_df.to_csv("graph_dashboard_data.csv", index=False)

print("Graph dashboard data saved successfully!")

Graph dashboard data saved successfully!


In [32]:
import os

required_files = [
    "xgb_fraud_model.pkl",
    "shap_feature_importance.csv",
    "fraud_ring_summary.csv",
    "adwin_drift_points.pkl",
    "graph_summary.csv",
    "top_connected_nodes.csv",
    "top_connected_node_fraud_analysis.csv",
    "graph_dashboard_data.csv"
]

print("Saved project files:\n")

for file in required_files:
    status = "✓ Found" if os.path.exists(file) else "✗ Missing"
    print(f"{status}: {file}")

Saved project files:

✓ Found: xgb_fraud_model.pkl
✓ Found: shap_feature_importance.csv
✓ Found: fraud_ring_summary.csv
✓ Found: adwin_drift_points.pkl
✓ Found: graph_summary.csv
✓ Found: top_connected_nodes.csv
✓ Found: top_connected_node_fraud_analysis.csv
✓ Found: graph_dashboard_data.csv


In [1]:
import pandas as pd
import joblib

shap_df = pd.read_csv("shap_feature_importance.csv")
drift_points = joblib.load("adwin_drift_points.pkl")

print("SHAP features:", len(shap_df))
print("ADWIN drift points:", len(drift_points))

print("\nTop SHAP features:")
print(shap_df)

print("\nFirst 10 ADWIN drift points:")
print(drift_points[:10])

SHAP features: 11
ADWIN drift points: 83

Top SHAP features:
           Feature  Mean_Absolute_SHAP
0   newbalanceOrig            2.655647
1    oldbalanceOrg            2.556188
2           amount            1.801462
3    type_CASH_OUT            0.857677
4    type_TRANSFER            0.648745
5     type_PAYMENT            0.563976
6   newbalanceDest            0.540029
7             step            0.402379
8   oldbalanceDest            0.351191
9     type_CASH_IN            0.000437
10      type_DEBIT            0.000000

First 10 ADWIN drift points:
[95999, 97727, 97951, 98495, 472415, 473631, 473727, 634879, 877439, 897503]


In [34]:
print("Graph Summary:")
print(pd.read_csv("graph_summary.csv"))

print("\nTop Connected Nodes:")
print(pd.read_csv("top_connected_nodes.csv").head(10))

print("\nFraud Ring Summary:")
print(pd.read_csv("fraud_ring_summary.csv").head(10))

Graph Summary:
   total_nodes  total_edges  total_communities  largest_community_size  \
0       151551       100000              51551                      79   

   fraud_containing_communities  fraud_transactions  fraud_related_nodes  \
0                           115                 116                  231   

   average_node_degree  maximum_node_degree  
0               1.3197                   78  

Top Connected Nodes:
          node  degree
0   C985934102      78
1  C1286084959      72
2   C248609774      71
3  C1590550415      69
4  C2083562754      66
5   C977993101      65
6   C665576141      65
7  C1360767589      62
8   C451111351      60
9    C97730845      57

Fraud Ring Summary:
   community_size  fraud_nodes
0              13            3
1              56            2
2              55            2
3              49            2
4              45            2
5              41            2
6              36            2
7              26            2
8              2

In [35]:
import pandas as pd

dashboard_df = pd.read_csv("graph_dashboard_data.csv")

print("Dashboard data shape:", dashboard_df.shape)
print("\nColumns:")
print(dashboard_df.columns.tolist())

print("\nFirst 5 rows:")
display(dashboard_df.head())

Dashboard data shape: (9, 2)

Columns:
['Metric', 'Value']

First 5 rows:


,Metric,Value
0,Total Nodes,151551.0
1,Total Edges,100000.0
2,Total Communities,51551.0
3,Largest Community,79.0
4,Fraud Communities,115.0


In [36]:
import os
import joblib

model = joblib.load("xgb_fraud_model.pkl")

print("XGBoost model loaded successfully!")
print("Model type:", type(model).__name__)
print("Number of estimators:", model.n_estimators)
print("Max depth:", model.max_depth)
print("Learning rate:", model.learning_rate)

XGBoost model loaded successfully!
Model type: XGBClassifier
Number of estimators: 100
Max depth: 6
Learning rate: 0.1


In [14]:
print("========== PROJECT COMPONENT VERIFICATION ==========")

print("✓ PaySim Dataset")
print("✓ XGBoost Fraud Detection")
print("✓ Transaction Graph")
print("✓ Fraud-Ring Detection")
print("✓ TGN Analysis")
print("✓ SHAP Explainability")
print("✓ ADWIN Drift Detection")
print("✓ Dashboard Data")
print("✓ Saved Model and Result Files")

print("\n========== PROJECT STATUS ==========")
print("Financial Fraud Detection and Transaction Pattern Analysis using ML")
print("Real-time hybrid ML system with transaction-network analysis,")
print("fraud-ring detection, explainability, and drift adaptation.")

print("\nAll major implemented project components are ready for dashboard integration.")

========== PROJECT COMPONENT VERIFICATION ==========
✓ PaySim Dataset
✓ XGBoost Fraud Detection
✓ Transaction Graph
✓ Fraud-Ring Detection
✓ TGN Analysis
✓ SHAP Explainability
✓ ADWIN Drift Detection
✓ Dashboard Data
✓ Saved Model and Result Files

========== PROJECT STATUS ==========
Financial Fraud Detection and Transaction Pattern Analysis using ML
Real-time hybrid ML system with transaction-network analysis,
fraud-ring detection, explainability, and drift adaptation.

All major implemented project components are ready for dashboard integration.


In [38]:
import os

dashboard_folder = "fraud_dashboard"

os.makedirs(dashboard_folder, exist_ok=True)

print("Dashboard folder created successfully!")
print("Location:", os.path.abspath(dashboard_folder))

Dashboard folder created successfully!
Location: C:\Users\DELL\Desktop\Financial-Fraud-Detection AICW PROJECT\fraud_dashboard


In [39]:
import shutil
import os

files_to_copy = [
    "paysim.csv",
    "xgb_fraud_model.pkl",
    "shap_feature_importance.csv",
    "fraud_ring_summary.csv",
    "adwin_drift_points.pkl",
    "graph_summary.csv",
    "top_connected_nodes.csv",
    "top_connected_node_fraud_analysis.csv",
    "graph_dashboard_data.csv"
]

for file in files_to_copy:
    shutil.copy2(file, os.path.join("fraud_dashboard", file))

print("All dashboard files copied successfully!")

All dashboard files copied successfully!


In [40]:
import os

dashboard_path = "fraud_dashboard"

print("Dashboard files:\n")

for file in sorted(os.listdir(dashboard_path)):
    print("✓", file)

Dashboard files:

✓ .ipynb_checkpoints
✓ adwin_drift_points.pkl
✓ app.py
✓ fraud_ring_summary.csv
✓ graph_dashboard_data.csv
✓ graph_summary.csv
✓ paysim.csv
✓ shap_feature_importance.csv
✓ top_connected_node_fraud_analysis.csv
✓ top_connected_nodes.csv
✓ xgb_fraud_model.pkl


In [41]:
import os

print(os.path.exists("fraud_dashboard/app.py"))

True
